# 02 · Contribuições e saldos das edições acadêmicas

O recorte histórico é composto por 1.021 revisões de 177 participações. O modo padrão usa a lista de revisões já identificadas, para preservar o recorte da pesquisa. As funções de API permitem consultar os mesmos `revid` e `parentid` ou pesquisar revisões de um usuário, quando necessário.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
BRUTOS = ROOT / 'dados' / 'brutos'
PROCESSADOS = ROOT / 'dados' / 'processados'
GRAFICOS = ROOT / 'resultados' / 'graficos'
PROCESSADOS.mkdir(parents=True, exist_ok=True)
GRAFICOS.mkdir(parents=True, exist_ok=True)

import requests, re, time, unicodedata
from collections import Counter

BASE = pd.read_excel(PROCESSADOS / 'base_participacoes_consolidada.xlsx', sheet_name='Base_Consolidada')
FONTE_HISTORICA = BRUTOS / 'resultado_tcc_contribuicoes_ampliado.xlsx'
detalhes = pd.read_excel(FONTE_HISTORICA, sheet_name='Detalhes_Edicoes')
participacoes = pd.read_excel(FONTE_HISTORICA, sheet_name='Participacoes_Aluno')
grupos = pd.read_excel(FONTE_HISTORICA, sheet_name='Grupos_Verbetes')
assert len(BASE)==243 and len(detalhes)==1021 and len(participacoes)==177 and len(grupos)==70

## Requisições à API MediaWiki

Informe seu e-mail no User-Agent antes de coletar. O tratamento de respostas 429 e erros temporários evita requisições excessivas.

In [ ]:
USER_AGENT = 'TCC-Wikipedia-Qualidade/1.0 (contato: seu-email@exemplo.com)'
sessao = requests.Session()
sessao.headers.update({'User-Agent': USER_AGENT})

def chamada_api(wiki='pt', **params):
    url = f'https://{wiki}.wikipedia.org/w/api.php'
    base = {'format':'json', 'formatversion':2, 'action':'query', **params}
    for tentativa in range(6):
        try:
            resposta = sessao.get(url, params=base, timeout=45)
            if resposta.status_code in (429, 500, 502, 503, 504):
                time.sleep(min(60, 2 ** tentativa))
                continue
            resposta.raise_for_status()
            payload = resposta.json()
            if 'error' in payload:
                raise RuntimeError(payload['error'])
            return payload
        except requests.RequestException:
            if tentativa == 5:
                raise
            time.sleep(min(60, 2 ** tentativa))
    raise RuntimeError('A consulta não foi concluída.')

def contribuicoes_usuario(usuario, wiki='pt'):
    """Lista revisões do usuário no domínio principal, com paginação."""
    registros = []
    continuar = {}
    while True:
        res = chamada_api(wiki,
            list='usercontribs', ucuser=usuario, ucnamespace=0,
            ucprop='ids|title|timestamp|sizediff', uclimit='max', **continuar)
        registros.extend(res.get('query', {}).get('usercontribs', []))
        continuar = res.get('continue')
        if not continuar:
            break
    return registros

def revisoes_por_id(ids, wiki='pt'):
    """Recupera wikitext e metadados de revisões históricas fixadas."""
    ids = sorted({int(x) for x in ids if pd.notna(x) and int(x)>0})
    saida = {}
    for inicio in range(0, len(ids), 40):
        lote = ids[inicio:inicio+40]
        res = chamada_api(wiki, prop='revisions', revids='|'.join(map(str,lote)),
            rvprop='ids|timestamp|size|content', rvslots='main')
        for pagina in res.get('query',{}).get('pages',[]):
            for r in pagina.get('revisions',[]):
                saida[int(r['revid'])] = {
                    'revid':int(r['revid']), 'parentid':int(r.get('parentid',0)),
                    'timestamp':r.get('timestamp'), 'bytes_total':int(r.get('size',0)),
                    'wikitext':r.get('slots',{}).get('main',{}).get('content','')
                }
        time.sleep(.1)
    return saida

## Métricas calculadas no wikitext

A contagem de referências procura aberturas `<ref>`. Links externos são URLs fora dessas marcações. Os resultados dependem dessas regras de detecção; não representam avaliação semântica do conteúdo.

In [ ]:
REF = re.compile(r'<ref\b[^>]*>', re.I)
REF_BLOCK = re.compile(r'<ref\b[^>]*>.*?</ref\s*>|<ref\b[^>]*/>', re.I|re.S)
SECAO = re.compile(r'^={2,6}\s*[^=\n]+?\s*={2,6}\s*$', re.M)
LINK_INTERNO = re.compile(r'\[\[([^]\n]+)\]\]')
LINK_EXTERNO = re.compile(r'https?://[^\s\]<>|{}]+', re.I)
CATEGORIA = re.compile(r'\[\[\s*Categoria\s*:', re.I)
IMAGEM = re.compile(r'\[\[\s*(?:Arquivo|Ficheiro|File|Image)\s*:', re.I)
INFOBOX = re.compile(r'\{\{\s*(?:Infobox|Info/)', re.I)
VER_TAMBEM = re.compile(r'^={2,6}\s*Ver também\s*={2,6}\s*$', re.I|re.M)
TABELA = re.compile(r'\{\|')

def medir_wikitext(texto, bytes_total=None):
    txt = texto or ''
    sem_ref = REF_BLOCK.sub('', txt)
    links = LINK_INTERNO.findall(txt)
    internos = [x for x in links if not re.match(r'\s*(?:Categoria|Arquivo|Ficheiro|File|Image)\s*:', x, flags=re.I)]
    return {
        'bytes_total': int(bytes_total) if bytes_total is not None else len(txt.encode('utf-8')),
        'referencias':len(REF.findall(txt)),
        'secoes':len(SECAO.findall(txt)),
        'infobox':int(bool(INFOBOX.search(txt))),
        'ver_tambem':int(bool(VER_TAMBEM.search(txt))),
        'links_internos':len(internos),
        'links_externos':len(LINK_EXTERNO.findall(sem_ref)),
        'categorias':len(CATEGORIA.findall(txt)),
        'imagens':len(IMAGEM.findall(txt)),
        'tabelas':len(TABELA.findall(txt)),
    }

print('Exemplo de leitura:', medir_wikitext('== História ==\nTexto<ref>Fonte</ref> [[Brasil]]'))

## Revisões fixadas e cálculo de saldos

Ative `CONSULTAR_API` quando quiser refazer as leituras de conteúdo. O padrão mantém o arquivo congelado que fundamentou as análises; a recaptura pode revelar diferenças nas regras de extração ou revisões indisponíveis.

In [ ]:
CONSULTAR_API = False  # True: consulta as revisões históricas identificadas na pesquisa
if CONSULTAR_API:
    assert 'seu-email@exemplo.com' not in USER_AGENT, 'Informe um contato no User-Agent.'
    ids = set(detalhes['revid'].dropna().astype(int)) | set(detalhes['parentid'].dropna().astype(int))
    revisoes = revisoes_por_id(ids, 'pt')
    recalculadas=[]
    for r in detalhes.itertuples(index=False):
        rev = revisoes.get(int(r.revid))
        anterior = revisoes.get(int(r.parentid)) if int(r.parentid)>0 else None
        if not rev:
            continue
        agora = medir_wikitext(rev['wikitext'], rev['bytes_total'])
        antes = medir_wikitext(anterior['wikitext'], anterior['bytes_total']) if anterior else medir_wikitext('',0)
        linha={'revid':int(r.revid), 'parentid':int(r.parentid)}
        for metrica in agora:
            linha[f'{metrica}_antes'] = antes[metrica]
            linha[f'{metrica}_depois'] = agora[metrica]
            linha[f'{metrica}_delta'] = agora[metrica] - antes[metrica]
        recalculadas.append(linha)
    reprocessadas = pd.DataFrame(recalculadas)
    reprocessadas.to_excel(PROCESSADOS/'auditoria_revisoes_recapturadas.xlsx',index=False)
    print('Revisões recapturadas:',len(reprocessadas))

## Nova busca por usuário e verbete (opcional)

O bloco abaixo permite descobrir edições pela API a partir do cadastro consolidado. Ele é desligado por padrão porque o histórico já foi fixado nos identificadores das 1.021 revisões. Um novo levantamento pode incluir edições feitas depois das atividades, verbetes renomeados ou revisões indisponíveis; por isso sua saída é separada e precisa de auditoria.

In [ ]:
NOVA_BUSCA_USUARIOS = False

def titulo_comparavel(valor):
    texto = unicodedata.normalize('NFKC', str(valor)).replace('_',' ').strip()
    return re.sub(r'\s+', ' ', texto).casefold()

if NOVA_BUSCA_USUARIOS:
    assert 'seu-email@exemplo.com' not in USER_AGENT, 'Informe um contato no User-Agent.'
    cache_usuarios = {}
    linhas_edicoes, status_consultas = [], []
    for cadastro in BASE.to_dict('records'):
        wiki = str(cadastro.get('wiki_api','pt')).strip()
        usuario = str(cadastro.get('usuario_wikipedia','')).strip()
        titulo = titulo_comparavel(cadastro.get('verbete',''))
        chave_usuario = (wiki, usuario)
        if chave_usuario not in cache_usuarios:
            cache_usuarios[chave_usuario] = contribuicoes_usuario(usuario, wiki=wiki)
        historico_usuario = cache_usuarios[chave_usuario]
        revisoes_encontradas = [e for e in historico_usuario
                                if titulo_comparavel(e.get('title',''))==titulo]
        situacao = ('ok' if revisoes_encontradas else
                    'sem_contribuicoes_usuario' if not historico_usuario else
                    'sem_contribuicao_no_verbete')
        status_consultas.append({'id_participacao':cadastro['id_participacao'],
                                 'status_api':situacao,'n_edicoes':len(revisoes_encontradas)})
        for ed in revisoes_encontradas:
            linhas_edicoes.append({
                'id_participacao':cadastro['id_participacao'],
                'usuario_wikipedia':usuario, 'verbete':cadastro.get('verbete'),
                'wiki_api':wiki, 'revid':int(ed['revid']),
                'parentid':int(ed.get('parentid',0)),
                'timestamp':ed.get('timestamp'), 'sizediff_api':ed.get('sizediff')
            })
    novos_detalhes = pd.DataFrame(linhas_edicoes)
    # As métricas são calculadas por diferença entre a revisão do estudante e seu parentid.
    cache_revisoes = {}
    for wiki, bloco in novos_detalhes.groupby('wiki_api'):
        ids = set(bloco['revid']) | set(bloco['parentid'])
        cache_revisoes[wiki] = revisoes_por_id(ids,wiki=wiki)
    linhas_metricas=[]
    for registro in novos_detalhes.to_dict('records'):
        cache=cache_revisoes[registro['wiki_api']]
        rev=cache.get(registro['revid'])
        parent=cache.get(registro['parentid']) if registro['parentid'] else None
        if rev is None:
            continue
        atual=medir_wikitext(rev['wikitext'],rev['bytes_total'])
        anterior=medir_wikitext(parent['wikitext'],parent['bytes_total']) if parent else medir_wikitext('',0)
        linha={**registro}
        for nome in atual:
            linha[f'{nome}_antes']=anterior[nome]
            linha[f'{nome}_depois']=atual[nome]
            linha[f'{nome}_delta']=atual[nome]-anterior[nome]
        linhas_metricas.append(linha)
    recaptura=pd.DataFrame(linhas_metricas)
    with pd.ExcelWriter(PROCESSADOS/'auditoria_busca_api_nova.xlsx',engine='openpyxl') as gravador:
        pd.DataFrame(status_consultas).to_excel(gravador,sheet_name='Status_Consultas',index=False)
        recaptura.to_excel(gravador,sheet_name='Revisoes_Recapturadas',index=False)
    print('Nova coleta gravada para auditoria:',len(recaptura),'revisões.')

## Conferência e saída histórica

Os saldos por participação são a soma dos deltas das edições do mesmo aluno no mesmo verbete. Não se somam atuações de um aluno em verbetes diferentes.

In [ ]:
for campo in ['bytes_saldo','referencias_saldo','secoes_saldo','links_internos_saldo','imagens_saldo']:
    assert campo in participacoes.columns
assert len(detalhes)==1021
assert participacoes['id_participacao'].is_unique
assert len(participacoes)==177

# Conferência com a soma dos deltas preservados na planilha histórica
colunas = [('referencias','referencias_saldo'),('secoes','secoes_saldo'),
           ('links_internos','links_internos_saldo'),('imagens','imagens_saldo')]
for delta, saldo in colunas:
    calculado = detalhes.groupby('id_participacao')[f'{delta}_delta'].sum()
    comparacao = participacoes.set_index('id_participacao')[saldo]
    assert (calculado.reindex(comparacao.index).fillna(0).round(6) == comparacao.round(6)).all(), delta

saida = PROCESSADOS/'resultado_tcc_contribuicoes_ampliado.xlsx'
with pd.ExcelWriter(saida, engine='openpyxl') as writer:
    detalhes.to_excel(writer,sheet_name='Detalhes_Edicoes',index=False)
    participacoes.to_excel(writer,sheet_name='Participacoes_Aluno',index=False)
    grupos.to_excel(writer,sheet_name='Grupos_Verbetes',index=False)
print('Arquivo gerado:',saida.name)